# Lab 10.3 &mdash; Read the Trace

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Read a ReAct trace as (tool, arg, observation) steps
- Find the step where a wrong/unknown tool was called
- Catch an ungrounded argument the final answer hides

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; guardrail logic, tool wiring, agent structure, and reading a fixed real message trace &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Read the trace — a broken run](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-10-03"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
The trace is your **#1 debugging surface** (deck slide 14). The final answer often looks plausible; the
trace shows **where and why** a run went wrong &mdash; a **wrong tool** at one step, an **ungrounded
argument** at another. You debug an agent by reading its reasoning like a transcript. Here you localise
two classic bugs from a recorded trace.

In [ ]:
# A broken run, recorded as steps of (tool, arg, observation).
TRACE = [
    ("lookup_order", "revenue", "unknown tool: lookup_order"),   # wrong tool
    ("compute", "0.15 * 100", 15.0),                             # 100 was never grounded!
]
GROUNDED = {"120"}   # the only value actually retrieved from the report was 120
print("steps:", len(TRACE), "| grounded values:", GROUNDED)

## Your Turn
Complete `find_wrong_tool`, `used_tools`, and `find_ungrounded`.

In [ ]:
def find_wrong_tool(steps):
    # index of the first step whose observation reports an unknown tool, else -1
    for i, (tool, arg, obs) in enumerate(steps):
        if ___:   # TODO: the observation string mentions an unknown tool
            return i
    return -1

def used_tools(steps):
    return ___   # TODO: the tool name of each step, in order

def find_ungrounded(steps, grounded):
    # a compute step whose argument uses a number that was never grounded
    for i, (tool, arg, obs) in enumerate(steps):
        if tool == "compute" and not any(g in str(arg) for g in grounded):   # TODO: keep
            return i
    return -1

try:
    print('wrong tool at step:', find_wrong_tool(TRACE))
    print('tools used        :', used_tools(TRACE))
    print('ungrounded at step:', find_ungrounded(TRACE, GROUNDED))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("locates the wrong-tool step", lambda: find_wrong_tool(TRACE) == 0)
expect_true("returns -1 when every tool is known", lambda: find_wrong_tool([("compute", "1+1", 2.0)]) == -1)
expect_true("lists the tools in order", lambda: used_tools(TRACE) == ["lookup_order", "compute"])
expect_true("catches the ungrounded argument (100 not grounded)", lambda: find_ungrounded(TRACE, {"120"}) == 1)
expect_true("a grounded compute is clean", lambda: find_ungrounded([("compute", "0.15 * 120", 18.0)], {"120"}) == -1)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
The trace shows not just THAT a run failed but WHERE and WHY -- a wrong tool at step 1, an ungrounded number at step 2. The final answer alone hides both. Transparency and debuggability are the same thing.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>